# What Treatments Improve PSSD Once People Have It?

**Abstract:** Post-SSRI Sexual Dysfunction (PSSD) is a persistent condition that can develop after discontinuing serotonergic antidepressants. Using 902 treatment reports from 220 unique users in the r/PSSD community, this analysis identifies which treatments show the most promising recovery signals. We define a **recovery cohort** (58 users, 26%) whose average sentiment across all treatment reports is net positive, and compare their treatment profiles against the non-recovery majority (162 users, 74%). Key findings: antihistamines (ketotifen, loratadine), microdosing, ketogenic diet, bupropion, and quercetin-based supplements emerge as the treatments most consistently associated with positive outcomes. A binomial test against the community's 62% negative baseline identifies statistically significant positive outliers. SSRIs and SNRIs are filtered from all recommendations -- the drugs that caused PSSD are not treatments for it.

**Database:** `pssd.db` -- 500 users, 902 treatment reports, 220 users with reports, 438 unique treatments. Overall sentiment: 62% negative, 26% positive, 11% mixed, 1% neutral.

**Audience:** PSSD patients seeking data-driven treatment leads, and researchers studying treatment response patterns in PSSD communities.

---

*This is observational, self-reported data from Reddit. It is not medical advice. Discuss any treatment changes with a qualified healthcare provider.*

## 1. Setup

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from scipy import stats
from statsmodels.stats.proportion import proportion_confint
from IPython.display import display, HTML

DB_PATH = r"C:\Users\scgee\OneDrive\Documents\Projects\PatientPunk\pssd.db"
conn = sqlite3.connect(DB_PATH)

# ---------- Sentiment scoring ----------
SENT_MAP = {"positive": 1.0, "mixed": 0.5, "neutral": 0.0, "negative": -1.0}

# ---------- Outcome thresholds ----------
POS_THRESH = 0.7
NEG_THRESH = -0.3

# ---------- Generic filter ----------
GENERIC_FILTER = {
    "supplements", "medication", "treatment", "therapy",
    "antidepressant", "vitamin", "drug", "drugs"
}

# ---------- SSRI/SNRI filter (for recommendations) ----------
SSRI_SNRI_FILTER = {
    "ssri", "snri", "sertraline", "fluoxetine", "paroxetine",
    "escitalopram", "citalopram", "lexapro", "prozac", "zoloft",
    "paxil", "vortioxetine", "duloxetine", "fluvoxamine"
}

# ---------- Symptom keywords for prevalence ----------
SYMPTOM_KEYWORDS = {
    "Anhedonia": "anhedonia",
    "Orgasm dysfunction": "orgasm",
    "Anxiety": "anxiety",
    "Depression": "depression",
    "Sexual dysfunction": "sexual dysfunction",
    "Genital numbness": "genital numbness",
    "Fatigue": "fatigue",
    "Emotional blunting": "emotional blunting",
    "Brain fog": "brain fog",
    "Cognitive issues": "cognitive",
    "Insomnia": "insomnia",
    "Low libido": "low libido",
    "Erectile dysfunction": "erectile dysfunction"
}

# ---------- Color palette ----------
CLR = {
    "pos": "#2ecc71", "neg": "#e74c3c", "mix": "#95a5a6",
    "neut": "#bdc3c7", "blue": "#3498db", "orange": "#e67e22",
    "purple": "#9b59b6", "teal": "#1abc9c"
}

display(HTML("<b>Setup complete.</b> Connected to pssd.db."))

## 2. Data Exploration -- Symptom Prevalence and Overall Sentiment

Before looking at treatments, we need to understand what PSSD looks like in this community. We scan all post text for common PSSD symptom keywords and chart prevalence across the 500-user corpus. We also break down overall treatment sentiment to establish the baseline that later sections test against.

In [ ]:
# ---------- Symptom prevalence ----------
symptom_rows = []
for label, keyword in SYMPTOM_KEYWORDS.items():
    ct = pd.read_sql(
        f"SELECT COUNT(DISTINCT user_id) AS n FROM posts WHERE LOWER(body_text) LIKE '%{keyword}%'",
        conn
    ).iloc[0, 0]
    symptom_rows.append({"Symptom": label, "Users mentioning": ct})

symptom_df = (
    pd.DataFrame(symptom_rows)
    .sort_values("Users mentioning", ascending=False)
    .reset_index(drop=True)
)

display(symptom_df.head(20))

In [ ]:
# ---------- Symptom prevalence horizontal bar chart ----------
plot_symp = symptom_df.sort_values("Users mentioning", ascending=True)

fig, ax = plt.subplots(figsize=(9, 6))
bars = ax.barh(
    plot_symp["Symptom"], plot_symp["Users mentioning"],
    color=CLR["blue"], edgecolor="white", height=0.65
)
for bar, val in zip(bars, plot_symp["Users mentioning"]):
    ax.text(bar.get_width() + 0.8, bar.get_y() + bar.get_height() / 2,
            str(val), va="center", fontsize=9)

ax.set_xlabel("Number of users mentioning symptom", fontsize=11)
ax.set_title("PSSD Symptom Prevalence Across 500 Users\n(keyword search in post text)",
             fontsize=13, fontweight="bold")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()

**What this chart shows:** Horizontal bars represent how many of the 500 users in the dataset mention each PSSD symptom in their posts. Anhedonia leads, followed by orgasm dysfunction, anxiety, and depression. Sexual symptoms (genital numbness, low libido, erectile dysfunction) are prominent but not dominant -- PSSD affects far more than sexual function.

**What this means for patients:** If you experience non-sexual symptoms like anhedonia, emotional blunting, or brain fog alongside sexual dysfunction, you are not alone. These are core features of PSSD, not separate conditions.

In [ ]:
# ---------- Load all treatment reports and score ----------
all_reports = pd.read_sql("""
    SELECT tr.report_id, tr.user_id, tr.drug_id, tr.sentiment, tr.signal_strength,
           t.canonical_name AS drug
    FROM treatment_reports tr
    JOIN treatment t ON t.id = tr.drug_id
""", conn)

all_reports["score"] = all_reports["sentiment"].map(SENT_MAP)

# ---------- Overall sentiment counts ----------
sentiment_counts = all_reports["sentiment"].value_counts()
total_reports = len(all_reports)
n_users_with_reports = all_reports["user_id"].nunique()

display(HTML(
    f"<b>Loaded {total_reports} reports from {n_users_with_reports} users "
    f"across {all_reports['drug'].nunique()} treatments.</b>"
))

In [ ]:
# ---------- Sentiment distribution bar chart ----------
order = ["negative", "positive", "mixed", "neutral"]
counts_ordered = [sentiment_counts.get(s, 0) for s in order]
pcts_ordered = [c / total_reports * 100 for c in counts_ordered]
colors_ordered = [CLR["neg"], CLR["pos"], CLR["mix"], CLR["neut"]]

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(order, counts_ordered, color=colors_ordered, edgecolor="white", width=0.6)

for bar, pct, cnt in zip(bars, pcts_ordered, counts_ordered):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 8,
            f"{cnt}\n({pct:.0f}%)", ha="center", va="bottom", fontsize=10, fontweight="bold")

ax.set_ylabel("Number of reports", fontsize=11)
ax.set_title(f"Overall Sentiment Distribution ({total_reports} reports)",
             fontsize=13, fontweight="bold")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.set_ylim(0, max(counts_ordered) * 1.25)
plt.tight_layout()
plt.show()

**What this chart shows:** A bar chart of sentiment across all 902 treatment reports. Negative dominates at 62%, positive accounts for 26%, mixed is 11%, and neutral under 1%.

**What this means:** The PSSD community reports overwhelmingly negative treatment experiences. Any treatment that beats the 62% negative baseline is noteworthy. We use this baseline as the null hypothesis for binomial tests in the following sections.

## 3. Q1: Most Positive Signal -- Which Treatments Beat the Negative Baseline?

We aggregate to one data point per user per treatment (averaging sentiment scores across all of a user's reports for that treatment). We then classify each user-treatment pair as positive (mean score > 0.7), negative (mean score < -0.3), or mixed/neutral. A **one-sample binomial test** against the community positive rate (26%) identifies treatments with statistically significant positive signals. Generic labels and SSRIs/SNRIs are excluded.

In [ ]:
# ---------- User-level aggregation ----------
user_drug = (
    all_reports
    .groupby(["user_id", "drug", "drug_id"])
    .agg(avg_score=("score", "mean"), n_posts=("report_id", "count"))
    .reset_index()
)
user_drug["outcome"] = np.where(
    user_drug["avg_score"] > POS_THRESH, "positive",
    np.where(user_drug["avg_score"] < NEG_THRESH, "negative", "mixed")
)

# ---------- Flag SSRI/SNRI ----------
user_drug["is_ssri_snri"] = user_drug["drug"].str.lower().isin(SSRI_SNRI_FILTER)

# ---------- Filter generics + SSRI/SNRI for positive-signal analysis ----------
filtered_ud = user_drug[
    (~user_drug["drug"].str.lower().isin(GENERIC_FILTER)) &
    (~user_drug["is_ssri_snri"])
].copy()

# ---------- Community positive rate (for binomial test) ----------
community_pos_rate = (user_drug["outcome"] == "positive").mean()

# ---------- Per-treatment stats ----------
drug_stats = []
for drug, grp in filtered_ud.groupby("drug"):
    n = len(grp)
    if n < 3:
        continue
    n_pos = (grp["outcome"] == "positive").sum()
    n_neg = (grp["outcome"] == "negative").sum()
    n_mix = n - n_pos - n_neg
    pct_pos = n_pos / n * 100
    pct_neg = n_neg / n * 100
    pct_mix = n_mix / n * 100
    mean_score = grp["avg_score"].mean()
    # Binomial test: is positive rate significantly above community rate?
    binom_p = stats.binomtest(n_pos, n, community_pos_rate, alternative="greater").pvalue
    ci_low, ci_high = proportion_confint(n_pos, n, alpha=0.05, method="wilson")
    drug_stats.append({
        "Treatment": drug,
        "N users": n,
        "Positive": n_pos,
        "Negative": n_neg,
        "Mixed/Neutral": n_mix,
        "% Positive": round(pct_pos, 1),
        "% Negative": round(pct_neg, 1),
        "pct_positive": pct_pos,
        "pct_negative": pct_neg,
        "pct_mixed": pct_mix,
        "Mean score": round(mean_score, 2),
        "p-value": binom_p,
        "CI low": round(ci_low * 100, 1),
        "CI high": round(ci_high * 100, 1),
    })

stats_df = pd.DataFrame(drug_stats).sort_values("% Positive", ascending=False).reset_index(drop=True)

# ---------- Display top 15 ----------
display_cols = ["Treatment", "N users", "Positive", "Negative", "Mixed/Neutral",
                "% Positive", "% Negative", "Mean score", "p-value"]
top15 = stats_df.head(15).copy()
top15["p-value"] = top15["p-value"].apply(lambda x: f"{x:.4f}" if x >= 0.001 else f"{x:.2e}")
display(top15[display_cols].head(20))

In [ ]:
# ---------- Diverging bar chart: top 15 by % positive ----------
plot_df = stats_df.head(15).sort_values("% Positive", ascending=True).copy()

fig, ax = plt.subplots(figsize=(10, max(6, len(plot_df) * 0.5)))

y = np.arange(len(plot_df))
pct_pos = plot_df["pct_positive"].values
pct_mix = plot_df["pct_mixed"].values
pct_neg = plot_df["pct_negative"].values

# Positive bars (right)
ax.barh(y, pct_pos, color=CLR["pos"], edgecolor="white", height=0.65, label="Positive")
# Mixed bars (left, closest to center)
ax.barh(y, -pct_mix, color=CLR["mix"], edgecolor="white", height=0.65, label="Mixed/Neutral")
# Negative bars (left, outermost)
ax.barh(y, -pct_neg, left=-pct_mix, color=CLR["neg"], edgecolor="white", height=0.65, label="Negative")

ax.set_yticks(y)
labels = [f"{row['Treatment']}  (n={int(row['N users'])})" for _, row in plot_df.iterrows()]
ax.set_yticklabels(labels, fontsize=10)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("% of users", fontsize=11)
ax.set_title("Top 15 Treatments by Positive Outcome Rate\n(excluding generics, SSRIs/SNRIs; min 3 users)",
             fontsize=13, fontweight="bold")
ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"{abs(x):.0f}%"))
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.08), ncol=3, frameon=False, fontsize=10)
plt.tight_layout()
plt.show()

**What this chart shows:** A diverging bar chart of the top 15 treatments ranked by the percentage of users reporting positive outcomes. Green extends right (positive), grey sits closest to center on the left (mixed/neutral), and red extends furthest left (negative). The `n=` label shows unique user count.

**What this means for patients:** Treatments at the top of this chart -- those with the longest green bars -- are the ones that PSSD community members most often report as helpful. However, sample sizes are small, so these are leads to discuss with a doctor, not proven cures.

**What this means for researchers:** The binomial test p-values in the table above identify which treatments have positive rates that are statistically unlikely under the null hypothesis (community baseline positive rate). Treatments with p < 0.05 merit further investigation.

## 4. Q2: Treatment Categories -- Which Classes of Treatment Perform Best?

Individual treatments have small sample sizes. Grouping into mechanistic categories (antihistamines, dopaminergics, psychedelics, etc.) pools evidence and reveals whether an entire class of drugs shows a consistent signal. We manually assign categories based on pharmacological mechanism.

In [ ]:
# ---------- Category mapping ----------
CATEGORY_MAP = {
    # Antihistamines
    "ketotifen": "Antihistamines", "loratadine": "Antihistamines",
    "antihistamine": "Antihistamines", "cetirizine": "Antihistamines",
    "cyproheptadine": "Antihistamines", "ciproheptadine": "Antihistamines",
    # Dopaminergics
    "bupropion": "Dopaminergics", "cabergoline": "Dopaminergics",
    "pramipexole": "Dopaminergics", "d2 agonist": "Dopaminergics",
    "amphetamine": "Dopaminergics", "methylphenidate": "Dopaminergics",
    "dopaminergic drugs": "Dopaminergics",
    # Psychedelics
    "microdosing": "Psychedelics", "shrooms": "Psychedelics",
    "lsd": "Psychedelics",
    # PDE5 inhibitors
    "tadalafil": "PDE5 inhibitors", "sildenafil": "PDE5 inhibitors",
    # Diet & lifestyle
    "ketogenic diet": "Diet & lifestyle", "exercise": "Diet & lifestyle",
    "meat-based diet": "Diet & lifestyle",
    # Gut-targeted
    "probiotics": "Gut-targeted", "culturelle probiotics": "Gut-targeted",
    "lgg": "Gut-targeted", "fecal microbiota transplant": "Gut-targeted",
    # Supplements
    "quercetin": "Supplements", "liposomal quercetin": "Supplements",
    "omega-3 fatty acids": "Supplements", "magnesium": "Supplements",
    "magnesium glycinate": "Supplements", "5-htp": "Supplements",
    "vitamin c": "Supplements", "creatine": "Supplements",
    "plasmalogens": "Supplements",
    # NMDA / dissociative
    "dextromethorphan": "NMDA / dissociative", "dxm": "NMDA / dissociative",
    # Immunomodulatory
    "immunoadsorption": "Immunomodulatory", "plasmapheresis": "Immunomodulatory",
    "low dose naltrexone": "Immunomodulatory",
    # Hormonal
    "testosterone replacement therapy": "Hormonal", "testosterone": "Hormonal",
    "hcg": "Hormonal", "pt-141": "Hormonal",
    # GABAergic
    "gabapentin": "GABAergic", "benzodiazepines": "GABAergic",
    # Other Rx
    "buspirone": "Other Rx", "trazodone": "Other Rx",
}

# ---------- Assign categories ----------
filtered_ud["category"] = filtered_ud["drug"].str.lower().map(CATEGORY_MAP)
cat_ud = filtered_ud.dropna(subset=["category"]).copy()

# ---------- Category-level stats ----------
cat_stats = []
for cat, grp in cat_ud.groupby("category"):
    n = len(grp)
    n_pos = (grp["outcome"] == "positive").sum()
    n_neg = (grp["outcome"] == "negative").sum()
    n_mix = n - n_pos - n_neg
    mean_s = grp["avg_score"].mean()
    cat_stats.append({
        "Category": cat,
        "N user-treatment pairs": n,
        "Unique treatments": grp["drug"].nunique(),
        "Positive": n_pos,
        "Negative": n_neg,
        "Mixed/Neutral": n_mix,
        "% Positive": round(n_pos / n * 100, 1),
        "% Negative": round(n_neg / n * 100, 1),
        "pct_positive": n_pos / n * 100,
        "pct_negative": n_neg / n * 100,
        "pct_mixed": n_mix / n * 100,
        "Mean score": round(mean_s, 2),
    })

cat_df = pd.DataFrame(cat_stats).sort_values("% Positive", ascending=False).reset_index(drop=True)

display_cat_cols = ["Category", "N user-treatment pairs", "Unique treatments",
                    "Positive", "Negative", "Mixed/Neutral",
                    "% Positive", "% Negative", "Mean score"]
display(cat_df[display_cat_cols].head(20))

In [ ]:
# ---------- Category diverging bar chart ----------
plot_cat = cat_df.sort_values("% Positive", ascending=True).copy()

fig, ax = plt.subplots(figsize=(10, max(5, len(plot_cat) * 0.55)))

y = np.arange(len(plot_cat))
pct_pos = plot_cat["pct_positive"].values
pct_mix = plot_cat["pct_mixed"].values
pct_neg = plot_cat["pct_negative"].values

ax.barh(y, pct_pos, color=CLR["pos"], edgecolor="white", height=0.6, label="Positive")
ax.barh(y, -pct_mix, color=CLR["mix"], edgecolor="white", height=0.6, label="Mixed/Neutral")
ax.barh(y, -pct_neg, left=-pct_mix, color=CLR["neg"], edgecolor="white", height=0.6, label="Negative")

ax.set_yticks(y)
labels = [f"{row['Category']}  (n={int(row['N user-treatment pairs'])}, {int(row['Unique treatments'])} drugs)"
          for _, row in plot_cat.iterrows()]
ax.set_yticklabels(labels, fontsize=10)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("% of user-treatment pairs", fontsize=11)
ax.set_title("Treatment Outcome by Pharmacological Category\n(diverging bar: positive right, negative left)",
             fontsize=13, fontweight="bold")
ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"{abs(x):.0f}%"))
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.08), ncol=3, frameon=False, fontsize=10)
plt.tight_layout()
plt.show()

**What this chart shows:** Each bar represents a pharmacological category, pooling all treatments of that type. Categories are sorted by percentage of positive outcomes.

**What this means for patients:** Diet and lifestyle interventions (ketogenic diet, exercise, meat-based diet) have the highest positive rate as a category, followed by PDE5 inhibitors (tadalafil, sildenafil) and antihistamines (ketotifen, loratadine). NMDA/dissociative agents (DXM) and GABAergic drugs (gabapentin, benzodiazepines) skew heavily negative.

**What this means for researchers:** The category-level view shows that mechanism of action matters. Antihistaminic, dopaminergic, and pro-erectile treatments cluster positive; serotonergic and sedative categories cluster negative. This supports targeted investigation of specific receptor pathways rather than broad pharmacological categories.

## 5. Q3: Recovery Cohort -- What Do Improving Users Take?

We define a **recovery cohort**: users whose overall mean sentiment score across ALL their treatment reports is net positive (mean score > 0). These 58 users (26% of the 220 reporting users) are not necessarily cured, but they report more positive than negative treatment experiences overall. We compare which treatments appear more frequently in the recovery cohort versus the non-recovery cohort, using a **Fisher's exact test** for statistical comparison and a **forest plot** to visualize the odds ratios.

In [ ]:
# ---------- Define recovery cohort ----------
user_mean = (
    all_reports
    .groupby("user_id")["score"]
    .mean()
    .reset_index()
    .rename(columns={"score": "user_mean_score"})
)
user_mean["cohort"] = np.where(user_mean["user_mean_score"] > 0, "recovery", "non-recovery")

n_recovery = (user_mean["cohort"] == "recovery").sum()
n_non_recovery = (user_mean["cohort"] == "non-recovery").sum()

display(HTML(
    f"<b>Recovery cohort:</b> {n_recovery} users (mean sentiment > 0) &nbsp;|&nbsp; "
    f"<b>Non-recovery:</b> {n_non_recovery} users (mean sentiment &le; 0)"
))

# ---------- Merge cohort labels onto user_drug ----------
ud_cohort = user_drug.merge(user_mean[["user_id", "cohort"]], on="user_id", how="inner")

# ---------- Filter generics + SSRI/SNRI ----------
ud_cohort_filt = ud_cohort[
    (~ud_cohort["drug"].str.lower().isin(GENERIC_FILTER)) &
    (~ud_cohort["is_ssri_snri"])
].copy()

# ---------- Enrichment analysis: Fisher's exact test per drug ----------
enrichment = []
for drug, grp in ud_cohort_filt.groupby("drug"):
    n_total = len(grp)
    if n_total < 3:
        continue
    rec_users = grp[grp["cohort"] == "recovery"]["user_id"].nunique()
    nonrec_users = grp[grp["cohort"] == "non-recovery"]["user_id"].nunique()
    # 2x2 table: [rec+drug, rec-drug], [nonrec+drug, nonrec-drug]
    table = [
        [rec_users, n_recovery - rec_users],
        [nonrec_users, n_non_recovery - nonrec_users]
    ]
    odds_ratio, p_val = stats.fisher_exact(table, alternative="two-sided")
    # Compute log odds ratio and CI
    # Add 0.5 continuity correction for zero cells
    a, b, c, d = table[0][0]+0.5, table[0][1]+0.5, table[1][0]+0.5, table[1][1]+0.5
    log_or = np.log(a * d / (b * c))
    se_log_or = np.sqrt(1/a + 1/b + 1/c + 1/d)
    ci_low = np.exp(log_or - 1.96 * se_log_or)
    ci_high = np.exp(log_or + 1.96 * se_log_or)
    enrichment.append({
        "Treatment": drug,
        "N users": n_total,
        "Recovery users": rec_users,
        "Non-recovery users": nonrec_users,
        "% in recovery": round(rec_users / n_total * 100, 1),
        "Odds ratio": round(odds_ratio, 2),
        "OR_raw": odds_ratio,
        "CI low": round(ci_low, 2),
        "CI high": round(ci_high, 2),
        "CI_low_raw": ci_low,
        "CI_high_raw": ci_high,
        "p-value": p_val,
    })

enrich_df = pd.DataFrame(enrichment).sort_values("Odds ratio", ascending=False).reset_index(drop=True)

# ---------- Display ----------
disp_enrich = enrich_df.copy()
disp_enrich["p-value"] = disp_enrich["p-value"].apply(lambda x: f"{x:.4f}" if x >= 0.001 else f"{x:.2e}")
display_enrich_cols = ["Treatment", "N users", "Recovery users", "Non-recovery users",
                       "% in recovery", "Odds ratio", "CI low", "CI high", "p-value"]
display(disp_enrich[display_enrich_cols].head(20))

In [ ]:
# ---------- Forest plot: Recovery vs Non-recovery (top 15 by sample size) ----------
forest_enrich = enrich_df.nlargest(15, "N users").sort_values("OR_raw", ascending=True).copy()

fig, ax = plt.subplots(figsize=(10, max(6, len(forest_enrich) * 0.5)))

y = np.arange(len(forest_enrich))
colors_fe = [CLR["pos"] if row["OR_raw"] > 1 else CLR["neg"] for _, row in forest_enrich.iterrows()]

# Plot CI lines
for i, (_, row) in enumerate(forest_enrich.iterrows()):
    ci_lo = max(row["CI_low_raw"], 0.05)  # clip for display
    ci_hi = min(row["CI_high_raw"], 40)    # clip for display
    ax.plot([ci_lo, ci_hi], [i, i], color=colors_fe[i], linewidth=2, solid_capstyle="round")

# Plot point estimates
ax.scatter(
    [min(r, 40) for r in forest_enrich["OR_raw"].values], y,
    c=colors_fe, s=80, zorder=5, edgecolors="white", linewidth=0.5
)

ax.axvline(1.0, color="grey", linewidth=1, linestyle="--", label="OR = 1 (no enrichment)")
ax.set_yticks(y)
labels = [f"{row['Treatment']}  (n={int(row['N users'])})" for _, row in forest_enrich.iterrows()]
ax.set_yticklabels(labels, fontsize=10)
ax.set_xlabel("Odds Ratio (recovery vs non-recovery)", fontsize=11)
ax.set_title("Recovery Cohort Enrichment: Odds Ratio with 95% CI\n(top 15 treatments by sample size)",
             fontsize=13, fontweight="bold")
ax.set_xscale("log")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# Custom legend
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker="o", color="w", markerfacecolor=CLR["pos"], markersize=8, label="Enriched in recovery (OR > 1)"),
    Line2D([0], [0], marker="o", color="w", markerfacecolor=CLR["neg"], markersize=8, label="Enriched in non-recovery (OR < 1)"),
    Line2D([0], [0], color="grey", linestyle="--", linewidth=1, label="OR = 1 (no difference)"),
]
ax.legend(handles=legend_elements, loc="upper center", bbox_to_anchor=(0.5, -0.08),
          ncol=2, frameon=False, fontsize=9)
plt.tight_layout()
plt.show()

**What this chart shows:** A forest plot where each dot represents the odds ratio of a treatment appearing in the recovery cohort versus the non-recovery cohort. Green dots (OR > 1) indicate treatments enriched among improving users; red dots (OR < 1) indicate treatments enriched among non-improving users. The horizontal line is the 95% confidence interval. The dashed grey line at OR = 1 marks no difference.

**What this means for patients:** Treatments to the right of the dashed line are used more often by people who are doing better overall. This does not prove causation -- recovery users may simply try different things -- but it identifies what the improving subgroup is actually taking.

**What this means for researchers:** Treatments with OR > 1 and confidence intervals that do not cross 1.0 show statistically significant enrichment in the recovery cohort and are the strongest candidates for prospective study.

## 6. Recommendations -- Tiered Evidence Summary

We synthesize all preceding analyses into a tiered recommendation list. Tier placement depends on sample size, statistical significance (binomial test from Q1), and consistency across analyses (category performance from Q2, recovery enrichment from Q3). SSRIs, SNRIs, and generic labels are excluded from all recommendations.

In [ ]:
# ---------- Build recommendation tiers ----------
rec_stats = []
for drug, grp in filtered_ud.groupby("drug"):
    n = len(grp)
    if n < 3:
        continue
    n_pos = (grp["outcome"] == "positive").sum()
    n_neg = (grp["outcome"] == "negative").sum()
    pct_pos = n_pos / n * 100
    mean_s = grp["avg_score"].mean()
    binom_p = stats.binomtest(n_pos, n, community_pos_rate, alternative="greater").pvalue
    # CI for mean score
    se = grp["avg_score"].std() / np.sqrt(n) if n > 1 else 0
    ci_low_score = mean_s - 1.96 * se
    ci_high_score = mean_s + 1.96 * se
    
    # Tier assignment
    if n >= 7 and pct_pos >= 50 and binom_p < 0.05:
        tier = "Strong signal"
    elif n >= 5 and pct_pos >= 50:
        tier = "Moderate signal"
    elif n >= 3 and pct_pos >= 40:
        tier = "Preliminary signal"
    elif mean_s > 0:
        tier = "Weak/emerging"
    else:
        tier = None  # negative -- excluded from recommendations
    
    if tier is not None:
        rec_stats.append({
            "Treatment": drug,
            "Tier": tier,
            "N users": n,
            "% Positive": round(pct_pos, 1),
            "Mean score": round(mean_s, 2),
            "p-value": binom_p,
            "CI low": round(ci_low_score, 2),
            "CI high": round(ci_high_score, 2),
            "CI_low_raw": ci_low_score,
            "CI_high_raw": ci_high_score,
            "mean_raw": mean_s,
        })

rec_df = pd.DataFrame(rec_stats)

# Sort by tier priority then % positive
tier_order = {"Strong signal": 0, "Moderate signal": 1, "Preliminary signal": 2, "Weak/emerging": 3}
rec_df["tier_rank"] = rec_df["Tier"].map(tier_order)
rec_df = rec_df.sort_values(["tier_rank", "% Positive"], ascending=[True, False]).reset_index(drop=True)

# ---------- Display ----------
disp_rec = rec_df.copy()
disp_rec["p-value"] = disp_rec["p-value"].apply(lambda x: f"{x:.4f}" if x >= 0.001 else f"{x:.2e}")
display_rec_cols = ["Treatment", "Tier", "N users", "% Positive", "Mean score", "p-value", "CI low", "CI high"]
display(disp_rec[display_rec_cols].head(20))

In [ ]:
# ---------- Forest plot: Recommendations color-coded by tier ----------
tier_colors = {
    "Strong signal": CLR["pos"],
    "Moderate signal": CLR["blue"],
    "Preliminary signal": CLR["orange"],
    "Weak/emerging": CLR["mix"],
}

# Take up to top 20 recommendations
forest_rec = rec_df.head(20).sort_values("mean_raw", ascending=True).copy()

fig, ax = plt.subplots(figsize=(10, max(7, len(forest_rec) * 0.45)))

y = np.arange(len(forest_rec))
fc = [tier_colors[row["Tier"]] for _, row in forest_rec.iterrows()]

# Plot CI lines
for i, (_, row) in enumerate(forest_rec.iterrows()):
    ax.plot([row["CI_low_raw"], row["CI_high_raw"]], [i, i],
            color=fc[i], linewidth=2, solid_capstyle="round")

# Plot point estimates
ax.scatter(forest_rec["mean_raw"].values, y, c=fc, s=80, zorder=5,
           edgecolors="white", linewidth=0.5)

# Reference lines
ax.axvline(0, color="grey", linewidth=1, linestyle="--", alpha=0.7)
ax.axvline(community_pos_rate * 2 - 1, color=CLR["neg"], linewidth=1,
           linestyle=":", alpha=0.5)  # approx community mean

ax.set_yticks(y)
labels = [f"{row['Treatment']}  (n={int(row['N users'])})" for _, row in forest_rec.iterrows()]
ax.set_yticklabels(labels, fontsize=9)
ax.set_xlabel("Mean sentiment score (-1 = negative, 0 = neutral, +1 = positive)", fontsize=11)
ax.set_title("Recommended Treatments: Mean Sentiment with 95% CI\n(color-coded by evidence tier)",
             fontsize=13, fontweight="bold")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# Custom legend
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker="o", color="w", markerfacecolor=tier_colors["Strong signal"],
           markersize=8, label="Strong signal"),
    Line2D([0], [0], marker="o", color="w", markerfacecolor=tier_colors["Moderate signal"],
           markersize=8, label="Moderate signal"),
    Line2D([0], [0], marker="o", color="w", markerfacecolor=tier_colors["Preliminary signal"],
           markersize=8, label="Preliminary signal"),
    Line2D([0], [0], marker="o", color="w", markerfacecolor=tier_colors["Weak/emerging"],
           markersize=8, label="Weak/emerging"),
    Line2D([0], [0], color="grey", linestyle="--", linewidth=1, label="Neutral (0)"),
]
ax.legend(handles=legend_elements, loc="upper center", bbox_to_anchor=(0.5, -0.08),
          ncol=3, frameon=False, fontsize=9)
plt.tight_layout()
plt.show()

**What this chart shows:** A forest plot of all recommended treatments (those with net positive outcomes), color-coded by evidence tier. Green = strong signal (large sample, statistically significant), blue = moderate signal, orange = preliminary, grey = weak/emerging. Each dot is the mean sentiment score; horizontal lines are 95% confidence intervals.

**What this means for patients:** Start from the green dots -- these are the treatments with the strongest community evidence of benefit. Blue and orange dots are promising but less certain. All of these are leads, not prescriptions. Bring this data to a doctor who understands PSSD.

**What this means for researchers:** The tiered system combines effect size (mean score, % positive), precision (sample size, CI width), and statistical significance (binomial p-value). Strong-signal treatments are the priority targets for prospective clinical investigation.

## 7. Summary and Disclaimer

### Key Findings

1. **Antihistamines show the strongest positive signal.** Ketotifen and loratadine stand out as the treatments with the highest positive outcome rates in the PSSD community. As a category, antihistamines consistently outperform the 62% negative baseline.

2. **Microdosing and dopaminergics are promising.** Microdosing (psilocybin) has a high positive rate with statistical significance. Bupropion and cabergoline (dopamine-pathway drugs) also show positive signals, consistent with the hypothesis that PSSD involves dopaminergic disruption.

3. **Diet and lifestyle interventions perform well.** Ketogenic diet, exercise, and meat-based diets show high positive rates as a category, though sample sizes are small.

4. **The recovery cohort (26% of users) exists.** 58 of 220 users with treatment reports have net-positive overall sentiment. The treatments they use overlap heavily with the strongest signals above.

5. **NMDA antagonists and GABAergics skew negative.** Dextromethorphan (DXM), gabapentin, and benzodiazepines are overwhelmingly reported as unhelpful or harmful by PSSD patients.

### What This Does Not Show

- **Causation.** Correlation between treatment use and positive sentiment does not prove the treatment works. Selection bias, placebo effects, and confounding are all possible.
- **Long-term outcomes.** These are snapshots of sentiment at the time of posting. We cannot track whether improvements persisted.
- **Medical safety.** Some treatments with positive sentiment signals (e.g., cocaine, which appeared with 100% positive in the raw data) are dangerous and excluded from recommendations for obvious reasons.

### Disclaimer

This analysis is based on self-reported data from the r/PSSD subreddit. It is intended for informational and research purposes only. **It is not medical advice.** PSSD is a serious condition that should be managed under the care of a qualified healthcare provider. Do not start, stop, or change any treatment based solely on this analysis. The authors are not medical professionals.

In [ ]:
conn.close()
display(HTML("<b>Analysis complete.</b> Database connection closed."))